<a href="https://colab.research.google.com/github/amzad-786githumb/Privacy-Preserving-Synthetic-Tabular-Data-Generation-Using-Generative-Adversarial-Networks/blob/main/04_Baseline_Deep_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Notebook 04
# Section A
# Install Required Packages
# ============================================================

# SDV Ecosystem
!pip -q install -U sdv ctgan copulas

# Deep Learning
!pip -q install -U torch torchvision torchaudio

# Utilities
!pip -q install -U pandas numpy scipy scikit-learn
!pip -q install -U matplotlib seaborn tqdm
!pip -q install -U pyyaml joblib

print("All required packages installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [1]:
# ============================================================
# Imports
# ============================================================

import os
import gc
import json
import yaml
import random
import warnings
import joblib

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [3]:
# ============================================================
# Project Root
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Project")

print(PROJECT_ROOT)

/content/drive/MyDrive/SPP_GAN_Project


In [4]:
# ============================================================
# Directory Structure
# ============================================================

DATASET_DIR = PROJECT_ROOT / "datasets"

RAW_DATA_DIR = DATASET_DIR / "raw"

PROCESSED_DIR = DATASET_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"

BASELINE_MODEL_DIR = MODELS_DIR / "baseline_models"

RESULT_DIR = PROJECT_ROOT / "results"

REPORT_DIR = PROJECT_ROOT / "reports"

CONFIG_DIR = PROJECT_ROOT / "configs"

LOG_DIR = PROJECT_ROOT / "logs"

FIGURE_DIR = PROJECT_ROOT / "figures"

for folder in [

    DATASET_DIR,

    RAW_DATA_DIR,

    PROCESSED_DIR,

    MODELS_DIR,

    BASELINE_MODEL_DIR,

    RESULT_DIR,

    REPORT_DIR,

    CONFIG_DIR,

    LOG_DIR,

    FIGURE_DIR

]:

    folder.mkdir(

        parents=True,

        exist_ok=True

    )

print("Project directories initialized.")

Project directories initialized.


In [5]:
# ============================================================
# Baseline Model Folders
# ============================================================

MODEL_DIRS = {

    "ctgan":

        BASELINE_MODEL_DIR / "ctgan",

    "tvae":

        BASELINE_MODEL_DIR / "tvae",

    "dp_ctgan":

        BASELINE_MODEL_DIR / "dp_ctgan",

    "ctabgan_plus":

        BASELINE_MODEL_DIR / "ctabgan_plus",

    "metadata":

        BASELINE_MODEL_DIR / "metadata"

}

for folder in MODEL_DIRS.values():

    folder.mkdir(

        parents=True,

        exist_ok=True

    )

print("Baseline folders created.")

Baseline folders created.


In [6]:
# ============================================================
# Random Seed
# ============================================================

RANDOM_STATE = 42

random.seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

torch.manual_seed(RANDOM_STATE)

torch.cuda.manual_seed_all(RANDOM_STATE)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

print(f"Random Seed : {RANDOM_STATE}")

Random Seed : 42


In [7]:
# ============================================================
# GPU
# ============================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else

    "cpu"

)

print("="*70)

print("Device :", DEVICE)

if DEVICE.type == "cuda":

    print(

        "GPU :",

        torch.cuda.get_device_name(0)

    )

    print(

        "CUDA Version :",

        torch.version.cuda

    )

    print(

        "Memory (GB):",

        round(

            torch.cuda.get_device_properties(0).total_memory

            /1024**3,

            2

        )

    )

else:

    print("Running on CPU")

print("="*70)

Device : cuda
GPU : Tesla T4
CUDA Version : 13.0
Memory (GB): 14.56


In [8]:
# ============================================================
# Dataset Verification
# ============================================================

DATASET_PATHS = {

    "Adult Income":

        PROCESSED_DIR /

        "adult_income_processed.csv",

    "Bank Marketing":

        PROCESSED_DIR /

        "bank_marketing_processed.csv",

    "Breast Cancer":

        PROCESSED_DIR /

        "breast_cancer_processed.csv"

}

missing = []

for dataset, path in DATASET_PATHS.items():

    if not path.exists():

        missing.append(path.name)

if missing:

    raise FileNotFoundError(

        f"Missing processed datasets : {missing}"

    )

print("All processed datasets found.")

All processed datasets found.


In [9]:
# ============================================================
# Version Information
# ============================================================

import sdv
import ctgan

versions = {

    "Python":

        os.sys.version.split()[0],

    "PyTorch":

        torch.__version__,

    "SDV":

        sdv.__version__,

    "CTGAN":

        ctgan.__version__,

    "Pandas":

        pd.__version__,

    "NumPy":

        np.__version__

}

version_df = pd.DataFrame(

    versions.items(),

    columns=[

        "Library",

        "Version"

    ]

)

display(version_df)

,Library,Version
0,Python,3.12.13
1,PyTorch,2.13.0+cu130
2,SDV,1.37.3
3,CTGAN,0.12.1
4,Pandas,3.0.3
5,NumPy,2.5.1


In [10]:
# ============================================================
# Save Global Configuration
# ============================================================

project_config = {

    "project": "SPP_GAN_Project",

    "random_seed": RANDOM_STATE,

    "device": DEVICE.type,

    "datasets": [

        "Adult Income",

        "Bank Marketing",

        "Breast Cancer"

    ],

    "baseline_models": [

        "CTGAN",

        "TVAE",

        "DP-CTGAN",

        "CTAB-GAN+"

    ]

}

with open(

    CONFIG_DIR /

    "project_config.json",

    "w"

) as f:

    json.dump(

        project_config,

        f,

        indent=4

    )

print("Global configuration exported.")

Global configuration exported.


In [11]:
# ============================================================
# Cleanup
# ============================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

print("Memory cleaned.")

Memory cleaned.


In [12]:
# ============================================================
# Verification
# ============================================================

print("="*80)

print("NOTEBOOK 04")

print("="*80)

print("Environment Ready")

print("Project Root :", PROJECT_ROOT)

print("Datasets Verified :", len(DATASET_PATHS))

print("Device :", DEVICE)

print()

print("Next")

print("Section B : Load Processed Datasets & Generate Metadata")

print("="*80)

NOTEBOOK 04
Environment Ready
Project Root : /content/drive/MyDrive/SPP_GAN_Project
Datasets Verified : 3
Device : cuda

Next
Section B : Load Processed Datasets & Generate Metadata


In [14]:
from pathlib import Path
import pandas as pd

DATASET_PATHS = {
    "Adult Income":
        PROCESSED_DIR / "adult_income_processed.csv",

    "Bank Marketing":
        PROCESSED_DIR / "bank_marketing_processed.csv",

    "Breast Cancer":
        PROCESSED_DIR / "breast_cancer_processed.csv"
}

datasets = {}

for name, path in DATASET_PATHS.items():

    if not path.exists():

        raise FileNotFoundError(path)

    datasets[name] = pd.read_csv(path)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [15]:
# ============================================================
# Dataset Overview
# ============================================================

overview = []

for dataset_name, dataframe in datasets.items():

    overview.append({

        "Dataset": dataset_name,

        "Rows": dataframe.shape[0],

        "Columns": dataframe.shape[1],

        "Missing Values": int(dataframe.isna().sum().sum()),

        "Duplicate Rows": int(dataframe.duplicated().sum())

    })

dataset_overview = pd.DataFrame(overview)

display(dataset_overview)

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Adult Income,48813,111,0,0
1,Bank Marketing,45211,44,0,0
2,Breast Cancer,569,31,0,0


In [16]:
# ============================================================
# Dataset Validation
# ============================================================

validation = []

for dataset_name, dataframe in datasets.items():

    validation.append({

        "Dataset": dataset_name,

        "Contains NaN":
            dataframe.isna().sum().sum() > 0,

        "Contains Duplicate":
            dataframe.duplicated().sum() > 0,

        "Empty Dataset":
            dataframe.empty,

        "Valid":
            (
                not dataframe.empty
                and dataframe.isna().sum().sum() == 0
            )

    })

validation_df = pd.DataFrame(validation)

display(validation_df)

,Dataset,Contains NaN,Contains Duplicate,Empty Dataset,Valid
0,Adult Income,False,False,False,True
1,Bank Marketing,False,False,False,True
2,Breast Cancer,False,False,False,True


In [17]:
# ============================================================
# Feature Detection
# ============================================================

feature_summary = {}

for dataset_name, dataframe in datasets.items():

    numeric = dataframe.select_dtypes(

        include=np.number

    ).columns.tolist()

    categorical = dataframe.select_dtypes(

        exclude=np.number

    ).columns.tolist()

    feature_summary[dataset_name] = {

        "numeric": numeric,

        "categorical": categorical

    }

print("Feature detection completed.")

Feature detection completed.


In [18]:
# ============================================================
# Generate Metadata
# ============================================================

from sdv.metadata import SingleTableMetadata

metadata = {}

for dataset_name, dataframe in datasets.items():

    meta = SingleTableMetadata()

    meta.detect_from_dataframe(data=dataframe)

    metadata[dataset_name] = meta

print("Metadata generated.")

Metadata generated.


In [19]:
# ============================================================
# Metadata Summary
# ============================================================

summary = []

for dataset_name, meta in metadata.items():

    column_dict = meta.to_dict()["columns"]

    summary.append({

        "Dataset": dataset_name,

        "Columns": len(column_dict),

        "Metadata Ready": True

    })

metadata_summary = pd.DataFrame(summary)

display(metadata_summary)

,Dataset,Columns,Metadata Ready
0,Adult Income,111,True
1,Bank Marketing,44,True
2,Breast Cancer,31,True


In [20]:
# ============================================================
# Validate Metadata
# ============================================================

validation_results = []

for dataset_name, dataframe in datasets.items():

    dataset_columns = list(dataframe.columns)

    metadata_columns = list(metadata[dataset_name].columns.keys())

    dataset_set = set(dataset_columns)

    metadata_set = set(metadata_columns)

    missing = sorted(dataset_set - metadata_set)

    extra = sorted(metadata_set - dataset_set)

    is_match = (

        len(missing) == 0

        and

        len(extra) == 0

    )

    validation_results.append({

        "Dataset": dataset_name,

        "Dataset Columns": len(dataset_columns),

        "Metadata Columns": len(metadata_columns),

        "Match": is_match,

        "Missing Columns": ", ".join(missing),

        "Extra Columns": ", ".join(extra)

    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

,Dataset,Dataset Columns,Metadata Columns,Match,Missing Columns,Extra Columns
0,Adult Income,111,111,True,,
1,Bank Marketing,44,44,True,,
2,Breast Cancer,31,31,True,,


In [21]:
# ============================================================
# Safety Check
# ============================================================

if not validation_df["Match"].all():

    raise ValueError(

        "Metadata does not match the processed datasets.\n"

        "Fix preprocessing before continuing."

    )

print("Metadata perfectly matches processed datasets.")

Metadata perfectly matches processed datasets.


In [22]:
# ============================================================
# Validate SDV Metadata
# ============================================================

for dataset_name in metadata:

    metadata[dataset_name].validate()

print("SDV validation completed.")

SDV validation completed.


In [25]:
# ============================================================
# Export Metadata
# ============================================================

METADATA_DIR = MODEL_DIRS["metadata"]

METADATA_DIR.mkdir(

    parents=True,

    exist_ok=True

)

for dataset_name in metadata:

    filename = (

        dataset_name.lower()

        .replace(" ", "_")

        + "_metadata.json"

    )

    metadata_path = METADATA_DIR / filename

    metadata[dataset_name].save_to_json(

        filepath=str(metadata_path),

        mode='overwrite'

    )

print("Fresh metadata exported.")

Fresh metadata exported.


In [27]:
# ============================================================
# Save Validation Report
# ============================================================

validation_report = (

    RESULT_DIR /

    "metadata_validation.csv"

)

summary_report = (

    RESULT_DIR /

    "metadata_summary.csv"

)

validation_df.to_csv(

    validation_report,

    index=False

)

summary = pd.DataFrame({

    "Dataset":[

        row["Dataset"]

        for row in validation_results

    ],

    "Rows":[

        len(datasets[row["Dataset"]])

        for row in validation_results

    ],

    "Columns":[

        len(datasets[row["Dataset"]].columns)

        for row in validation_results

    ]

})

summary.to_csv(

    summary_report,

    index=False

)

print("Validation reports saved.")

Validation reports saved.


In [28]:
METADATA_DIR = MODEL_DIRS["metadata"]

for dataset_name, meta in metadata.items():

    filename = (

        dataset_name
        .lower()
        .replace(" ", "_")
        + "_metadata.json"

    )

    meta.save_to_json(
        filepath=METADATA_DIR / filename,
        mode='overwrite'
    )

print("Metadata exported successfully.")

Metadata exported successfully.


In [29]:
# ============================================================
#  Dataset Statistics
# ============================================================

statistics = []

for dataset_name, dataframe in datasets.items():

    statistics.append({

        "Dataset": dataset_name,

        "Rows": len(dataframe),

        "Columns": len(dataframe.columns),

        "Numeric":

            len(

                feature_summary[dataset_name]["numeric"]

            ),

        "Categorical":

            len(

                feature_summary[dataset_name]["categorical"]

            )

    })

statistics_df = pd.DataFrame(statistics)

display(statistics_df)

,Dataset,Rows,Columns,Numeric,Categorical
0,Adult Income,48813,111,111,0
1,Bank Marketing,45211,44,44,0
2,Breast Cancer,569,31,31,0


In [30]:
# ============================================================
# Export Reports
# ============================================================

dataset_overview.to_csv(

    REPORT_DIR /

    "dataset_overview.csv",

    index=False

)

validation_df.to_csv(

    REPORT_DIR /

    "dataset_validation.csv",

    index=False

)

statistics_df.to_csv(

    REPORT_DIR /

    "dataset_statistics.csv",

    index=False

)

print("Reports exported.")

Reports exported.


In [31]:
# ============================================================
# Integrity Check
# ============================================================

integrity = True

for dataset_name, dataframe in datasets.items():

    if dataframe.empty:

        integrity = False

        print(f"{dataset_name} is empty.")

    if dataframe.isna().sum().sum() > 0:

        integrity = False

        print(f"{dataset_name} contains missing values.")

if integrity:

    print("All datasets passed integrity checks.")

All datasets passed integrity checks.


In [32]:
# ============================================================
# Verification
# ============================================================

print("="*80)

print("Notebook 04")
print("Section B Completed")

print("="*80)

print(f"Datasets Loaded : {len(datasets)}")

print(f"Metadata Objects : {len(metadata)}")

print(f"Reports Generated : 3")

print(f"Metadata Exported : {len(metadata)}")

print()

print("Next")

print("Section C : Unified Hyperparameter Registry")

print("="*80)

Notebook 04
Section B Completed
Datasets Loaded : 3
Metadata Objects : 3
Reports Generated : 3
Metadata Exported : 3

Next
Section C : Unified Hyperparameter Registry


In [33]:
# ============================================================
# Global Experiment Configuration
# ============================================================

GLOBAL_CONFIG = {

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------
    "random_state": RANDOM_STATE,

    # --------------------------------------------------------
    # Device
    # --------------------------------------------------------
    "device": DEVICE.type,

    # --------------------------------------------------------
    # General Training
    # --------------------------------------------------------
    "epochs": 200,

    "batch_size": 256,

    "learning_rate": 2e-4,

    "optimizer": "Adam",

    "weight_decay": 1e-6,

    "gradient_clip": 1.0,

    "patience": 30,

    "checkpoint_interval": 10,

    "mixed_precision": True,

    # --------------------------------------------------------
    # DataLoader
    # --------------------------------------------------------
    "shuffle": True,

    "drop_last": False,

    "pin_memory": torch.cuda.is_available(),

    "num_workers": 2

}

print("Global configuration created.")

Global configuration created.


In [34]:
# ============================================================
# CTGAN Configuration
# ============================================================

CTGAN_CONFIG = {

    "epochs": GLOBAL_CONFIG["epochs"],

    "batch_size": GLOBAL_CONFIG["batch_size"],

    "embedding_dim": 128,

    "generator_dim": (256, 256),

    "discriminator_dim": (256, 256),

    "generator_lr": GLOBAL_CONFIG["learning_rate"],

    "discriminator_lr": GLOBAL_CONFIG["learning_rate"],

    "generator_decay": 1e-6,

    "discriminator_decay": 1e-6,

    "discriminator_steps": 1,

    "verbose": True,

    # Important to avoid PAC assertion problems
    "pac": 1

}

print("CTGAN configuration ready.")

CTGAN configuration ready.


In [35]:
# ============================================================
# TVAE Configuration
# ============================================================

TVAE_CONFIG = {

    "epochs": GLOBAL_CONFIG["epochs"],

    "batch_size": GLOBAL_CONFIG["batch_size"],

    "embedding_dim": 128,

    "compress_dims": (256, 256),

    "decompress_dims": (256, 256),

    "loss_factor": 2,

    "learning_rate": GLOBAL_CONFIG["learning_rate"],

    "cuda": torch.cuda.is_available()

}

print("TVAE configuration ready.")

TVAE configuration ready.


In [36]:
# ============================================================
# DP-CTGAN Configuration
# ============================================================

DP_CTGAN_CONFIG = {

    **CTGAN_CONFIG,

    "privacy": {

        "enabled": True,

        "epsilon": 5.0,

        "delta": 1e-5,

        "max_grad_norm": 1.0,

        "noise_multiplier": 1.1

    }

}

print("DP-CTGAN configuration ready.")

DP-CTGAN configuration ready.


In [37]:
# ============================================================
# CTAB-GAN+ Configuration
# ============================================================

CTABGAN_CONFIG = {

    "epochs": GLOBAL_CONFIG["epochs"],

    "batch_size": GLOBAL_CONFIG["batch_size"],

    "latent_dim": 128,

    "generator_lr": GLOBAL_CONFIG["learning_rate"],

    "discriminator_lr": GLOBAL_CONFIG["learning_rate"],

    "classifier_lr": GLOBAL_CONFIG["learning_rate"],

    "hidden_dim": 256,

    "verbose": True

}

print("CTAB-GAN+ configuration ready.")

CTAB-GAN+ configuration ready.


In [38]:
# ============================================================
# Unified Configuration Registry
# ============================================================

MODEL_CONFIGS = {

    "CTGAN": CTGAN_CONFIG,

    "TVAE": TVAE_CONFIG,

    "DP_CTGAN": DP_CTGAN_CONFIG,

    "CTABGAN_PLUS": CTABGAN_CONFIG

}

print("="*70)

print("Registered Models")

for model in MODEL_CONFIGS:

    print("✓", model)

print("="*70)

Registered Models
✓ CTGAN
✓ TVAE
✓ DP_CTGAN
✓ CTABGAN_PLUS


In [39]:
# ============================================================
# Export Configuration Files
# ============================================================

MODEL_CONFIG_DIR = CONFIG_DIR / "baseline_models"

MODEL_CONFIG_DIR.mkdir(

    parents=True,

    exist_ok=True

)

for model_name, config in MODEL_CONFIGS.items():

    file = (

        MODEL_CONFIG_DIR /

        f"{model_name.lower()}.json"

    )

    with open(file, "w") as f:

        json.dump(config, f, indent=4)

print("All model configurations exported.")

All model configurations exported.


In [40]:
# ============================================================
# Configuration Summary
# ============================================================

summary = []

for model_name, config in MODEL_CONFIGS.items():

    summary.append({

        "Model": model_name,

        "Epochs": config["epochs"],

        "Batch Size": config["batch_size"],

        "Learning Rate": GLOBAL_CONFIG["learning_rate"]

    })

config_summary = pd.DataFrame(summary)

display(config_summary)

,Model,Epochs,Batch Size,Learning Rate
0,CTGAN,200,256,0.0002
1,TVAE,200,256,0.0002
2,DP_CTGAN,200,256,0.0002
3,CTABGAN_PLUS,200,256,0.0002


In [41]:
# ============================================================
# Validation
# ============================================================

assert len(MODEL_CONFIGS) == 4

assert "CTGAN" in MODEL_CONFIGS

assert "TVAE" in MODEL_CONFIGS

assert "DP_CTGAN" in MODEL_CONFIGS

assert "CTABGAN_PLUS" in MODEL_CONFIGS

print("="*70)

print("Configuration validation successful.")

print("="*70)

Configuration validation successful.


In [42]:
# ============================================================
# Section Verification
# ============================================================

print("="*80)

print("Notebook 04")

print("Section C Completed")

print("="*80)

print(f"Global Configuration : ✓")

print(f"Model Configurations : {len(MODEL_CONFIGS)}")

print(f"Configuration Files  : {len(list(MODEL_CONFIG_DIR.glob('*.json')))}")

print()

print("Next")

print("Section D : CTGAN Factory Functions")

print("="*80)

Notebook 04
Section C Completed
Global Configuration : ✓
Model Configurations : 4
Configuration Files  : 4

Next
Section D : CTGAN Factory Functions


In [43]:
# ============================================================
# CTGAN Imports
# ============================================================

from sdv.single_table import CTGANSynthesizer

print("CTGAN imported successfully.")

CTGAN imported successfully.


In [44]:
# ============================================================
# CTGAN Configuration Loader
# ============================================================

def get_ctgan_config():

    """
    Returns a copy of the CTGAN configuration.
    """

    return CTGAN_CONFIG.copy()


print("CTGAN configuration loader ready.")

CTGAN configuration loader ready.


In [45]:
# ============================================================
# CTGAN Factory
# ============================================================

def create_ctgan(metadata):

    """
    Create a fresh CTGAN synthesizer.

    Parameters
    ----------
    metadata : SingleTableMetadata

    Returns
    -------
    CTGANSynthesizer
    """

    config = get_ctgan_config()

    synthesizer = CTGANSynthesizer(

        metadata=metadata,

        epochs=config["epochs"],

        batch_size=config["batch_size"],

        embedding_dim=config["embedding_dim"],

        generator_dim=config["generator_dim"],

        discriminator_dim=config["discriminator_dim"],

        generator_lr=config["generator_lr"],

        discriminator_lr=config["discriminator_lr"],

        generator_decay=config["generator_decay"],

        discriminator_decay=config["discriminator_decay"],

        discriminator_steps=config["discriminator_steps"],

        verbose=config["verbose"],

        pac=config["pac"]

    )

    return synthesizer


print("CTGAN factory created.")

CTGAN factory created.


In [46]:
# ============================================================
#  Create CTGAN Models
# ============================================================

ctgan_models = {}

for dataset_name in datasets.keys():

    ctgan_models[dataset_name] = create_ctgan(

        metadata[dataset_name]

    )

print("CTGAN models created.")

print()

for model in ctgan_models:

    print(model)

CTGAN models created.

Adult Income
Bank Marketing
Breast Cancer


In [47]:
# ============================================================
# Cell 39 : CTGAN Information
# ============================================================

summary = []

for dataset_name in ctgan_models:

    summary.append({

        "Dataset": dataset_name,

        "Model": "CTGAN",

        "Epochs": CTGAN_CONFIG["epochs"],

        "Batch Size": CTGAN_CONFIG["batch_size"],

        "Embedding": CTGAN_CONFIG["embedding_dim"],

        "PAC": CTGAN_CONFIG["pac"]

    })

ctgan_summary = pd.DataFrame(summary)

display(ctgan_summary)

,Dataset,Model,Epochs,Batch Size,Embedding,PAC
0,Adult Income,CTGAN,200,256,128,1
1,Bank Marketing,CTGAN,200,256,128,1
2,Breast Cancer,CTGAN,200,256,128,1


In [48]:
# ============================================================
# Cell 40 : Validation
# ============================================================

assert len(ctgan_models) == len(datasets)

for dataset_name, model in ctgan_models.items():

    assert isinstance(

        model,

        CTGANSynthesizer

    )

print("="*70)

print("CTGAN validation successful.")

print("="*70)

CTGAN validation successful.


In [49]:
# ============================================================
# Cell 41 : Helper Functions
# ============================================================

def get_ctgan(dataset_name):

    """
    Return CTGAN synthesizer.

    """

    return ctgan_models[dataset_name]


def list_ctgan_models():

    """

    List available CTGAN models.

    """

    return list(ctgan_models.keys())


print("Helper functions created.")

Helper functions created.


In [50]:
# ============================================================
# Cell 42 : Verification
# ============================================================

print("="*80)

print("Notebook 04")

print("Section D Completed")

print("="*80)

print("CTGAN Models Created :", len(ctgan_models))

print("Datasets")

for dataset_name in ctgan_models:

    print("✓", dataset_name)

print()

print("Training : Not Started")

print("Saved Models : None")

print()

print("Next")

print("Section E : TVAE Factory")

print("="*80)

Notebook 04
Section D Completed
CTGAN Models Created : 3
Datasets
✓ Adult Income
✓ Bank Marketing
✓ Breast Cancer

Training : Not Started
Saved Models : None

Next
Section E : TVAE Factory


In [51]:
# ============================================================
# Cell 43 : TVAE Import
# ============================================================

from sdv.single_table import TVAESynthesizer

print("TVAE imported successfully.")

TVAE imported successfully.


In [52]:
# ============================================================
# Cell 44 : TVAE Configuration Loader
# ============================================================

def get_tvae_config():
    """
    Return a copy of the TVAE configuration.
    """
    return TVAE_CONFIG.copy()

print("TVAE configuration loader ready.")

TVAE configuration loader ready.


In [53]:
# ============================================================
# Cell 45 : TVAE Factory
# ============================================================

def create_tvae(metadata):
    """
    Create a fresh TVAE synthesizer.

    Parameters
    ----------
    metadata : SingleTableMetadata

    Returns
    -------
    TVAESynthesizer
    """

    config = get_tvae_config()

    synthesizer = TVAESynthesizer(

        metadata=metadata,

        epochs=config["epochs"],

        batch_size=config["batch_size"],

        embedding_dim=config["embedding_dim"],

        compress_dims=config["compress_dims"],

        decompress_dims=config["decompress_dims"],

        loss_factor=config["loss_factor"],

        cuda=torch.cuda.is_available()

    )

    return synthesizer


print("TVAE factory created.")

TVAE factory created.


In [54]:
# ============================================================
# Cell 46 : Create TVAE Models
# ============================================================

tvae_models = {}

for dataset_name in datasets.keys():

    tvae_models[dataset_name] = create_tvae(
        metadata[dataset_name]
    )

print("="*70)

print("TVAE Models Created")

print("="*70)

for model in tvae_models:
    print(model)

TVAE Models Created
Adult Income
Bank Marketing
Breast Cancer


In [55]:
# ============================================================
# Cell 47 : TVAE Summary
# ============================================================

summary = []

for dataset_name in tvae_models:

    summary.append({

        "Dataset": dataset_name,

        "Model": "TVAE",

        "Epochs": TVAE_CONFIG["epochs"],

        "Batch Size": TVAE_CONFIG["batch_size"],

        "Embedding Dimension": TVAE_CONFIG["embedding_dim"],

        "Compression Layers": str(TVAE_CONFIG["compress_dims"]),

        "Decompression Layers": str(TVAE_CONFIG["decompress_dims"])

    })

tvae_summary = pd.DataFrame(summary)

display(tvae_summary)

,Dataset,Model,Epochs,Batch Size,Embedding Dimension,Compression Layers,Decompression Layers
0,Adult Income,TVAE,200,256,128,"(256, 256)","(256, 256)"
1,Bank Marketing,TVAE,200,256,128,"(256, 256)","(256, 256)"
2,Breast Cancer,TVAE,200,256,128,"(256, 256)","(256, 256)"


In [56]:
# ============================================================
# Cell 48 : Validation
# ============================================================

assert len(tvae_models) == len(datasets)

for dataset_name, model in tvae_models.items():

    assert isinstance(
        model,
        TVAESynthesizer
    )

print("="*70)

print("TVAE validation successful.")

print("="*70)

TVAE validation successful.


In [57]:
# ============================================================
# Cell 49 : Helper Functions
# ============================================================

def get_tvae(dataset_name):
    """
    Return TVAE synthesizer.
    """
    return tvae_models[dataset_name]


def list_tvae_models():
    """
    List available TVAE models.
    """
    return list(tvae_models.keys())


print("TVAE helper functions created.")

TVAE helper functions created.


In [58]:
# ============================================================
# Cell 50 : Verification
# ============================================================

print("="*80)

print("Notebook 04")

print("Section E Completed")

print("="*80)

print("TVAE Models Created :", len(tvae_models))

print()

print("Datasets")

for dataset_name in tvae_models:
    print("✓", dataset_name)

print()

print("Training Status : Not Started")

print("Saved Models : None")

print()

print("Next")

print("Section F : DP-CTGAN Factory")

print("="*80)

Notebook 04
Section E Completed
TVAE Models Created : 3

Datasets
✓ Adult Income
✓ Bank Marketing
✓ Breast Cancer

Training Status : Not Started
Saved Models : None

Next
Section F : DP-CTGAN Factory


In [59]:
# ============================================================
# Cell 51 : Differential Privacy Configuration
# ============================================================

DP_CONFIG = {

    # --------------------------------------------------------
    # Privacy Parameters
    # --------------------------------------------------------

    "epsilon": 5.0,

    "delta": 1e-5,

    "max_grad_norm": 1.0,

    "noise_multiplier": 1.1,

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    "secure_mode": False,

    "poisson_sampling": True,

    "target_delta": 1e-5,

    "accountant": "rdp"

}

print("Differential Privacy configuration created.")

Differential Privacy configuration created.


In [60]:
# ============================================================
# Cell 52 : DP-CTGAN Configuration
# ============================================================

DP_CTGAN_FACTORY_CONFIG = {

    **CTGAN_CONFIG,

    **DP_CONFIG

}

print("DP-CTGAN configuration ready.")

DP-CTGAN configuration ready.


In [61]:
# ============================================================
# Cell 53 : Configuration Loader
# ============================================================

def get_dp_ctgan_config():

    """
    Return a copy of the DP-CTGAN configuration.
    """

    return DP_CTGAN_FACTORY_CONFIG.copy()

print("DP-CTGAN configuration loader ready.")

DP-CTGAN configuration loader ready.


In [62]:
# ============================================================
# Cell 54 : DP-CTGAN Factory
# ============================================================

def create_dp_ctgan(metadata):

    """
    Create CTGAN architecture that will later
    be trained using Differential Privacy.

    """

    config = get_dp_ctgan_config()

    synthesizer = CTGANSynthesizer(

        metadata=metadata,

        epochs=config["epochs"],

        batch_size=config["batch_size"],

        embedding_dim=config["embedding_dim"],

        generator_dim=config["generator_dim"],

        discriminator_dim=config["discriminator_dim"],

        generator_lr=config["generator_lr"],

        discriminator_lr=config["discriminator_lr"],

        generator_decay=config["generator_decay"],

        discriminator_decay=config["discriminator_decay"],

        discriminator_steps=config["discriminator_steps"],

        verbose=config["verbose"],

        pac=config["pac"]

    )

    return synthesizer

In [63]:
# ============================================================
# Cell 55 : Create DP-CTGAN Models
# ============================================================

dp_ctgan_models = {}

for dataset_name in datasets:

    dp_ctgan_models[dataset_name] = create_dp_ctgan(

        metadata[dataset_name]

    )

print("="*70)

print("DP-CTGAN Models Created")

print("="*70)

for model in dp_ctgan_models:

    print(model)

DP-CTGAN Models Created
Adult Income
Bank Marketing
Breast Cancer


In [64]:
# ============================================================
# Cell 56 : Privacy Summary
# ============================================================

summary = []

for dataset_name in dp_ctgan_models:

    summary.append({

        "Dataset": dataset_name,

        "Model": "DP-CTGAN",

        "Epsilon": DP_CONFIG["epsilon"],

        "Delta": DP_CONFIG["delta"],

        "Noise Multiplier": DP_CONFIG["noise_multiplier"],

        "Gradient Clip": DP_CONFIG["max_grad_norm"]

    })

dp_summary = pd.DataFrame(summary)

display(dp_summary)

,Dataset,Model,Epsilon,Delta,Noise Multiplier,Gradient Clip
0,Adult Income,DP-CTGAN,5.0,0.00001,1.1,1.0
1,Bank Marketing,DP-CTGAN,5.0,0.00001,1.1,1.0
2,Breast Cancer,DP-CTGAN,5.0,0.00001,1.1,1.0


In [65]:
# ============================================================
# Cell 57 : Validation
# ============================================================

assert len(dp_ctgan_models) == len(datasets)

for model in dp_ctgan_models.values():

    assert isinstance(

        model,

        CTGANSynthesizer

    )

print("="*70)

print("DP-CTGAN validation successful.")

print("="*70)

DP-CTGAN validation successful.


In [66]:
# ============================================================
# Cell 58 : Helper Functions
# ============================================================

def get_dp_ctgan(dataset_name):

    return dp_ctgan_models[dataset_name]


def list_dp_ctgan_models():

    return list(dp_ctgan_models.keys())


print("DP-CTGAN helper functions created.")

DP-CTGAN helper functions created.


In [67]:
# ============================================================
# Cell 59 : Verification
# ============================================================

print("="*80)

print("Notebook 04")

print("Section F Completed")

print("="*80)

print("DP-CTGAN Models :", len(dp_ctgan_models))

print()

print("Privacy Settings")

for key, value in DP_CONFIG.items():

    print(f"{key:<20} : {value}")

print()

print("Next")

print("Section G : CTAB-GAN+ Factory")

print("="*80)

Notebook 04
Section F Completed
DP-CTGAN Models : 3

Privacy Settings
epsilon              : 5.0
delta                : 1e-05
max_grad_norm        : 1.0
noise_multiplier     : 1.1
secure_mode          : False
poisson_sampling     : True
target_delta         : 1e-05
accountant           : rdp

Next
Section G : CTAB-GAN+ Factory


In [68]:
!pip list | grep -i ctab

In [69]:
# ============================================================
# Cell 60 : Unified Baseline Model Registry
# ============================================================

BASELINE_MODELS = {

    "CTGAN": {

        "factory": create_ctgan,

        "config": CTGAN_CONFIG,

        "models": ctgan_models,

        "description": "Conditional Tabular GAN"

    },

    "TVAE": {

        "factory": create_tvae,

        "config": TVAE_CONFIG,

        "models": tvae_models,

        "description": "Tabular Variational Autoencoder"

    },

    "DP_CTGAN": {

        "factory": create_dp_ctgan,

        "config": DP_CTGAN_FACTORY_CONFIG,

        "models": dp_ctgan_models,

        "description": "Differentially Private CTGAN"

    }

}

print("Unified baseline model registry created.")

Unified baseline model registry created.


In [70]:
# ============================================================
# Cell 61 : Dataset Registry
# ============================================================

DATASET_REGISTRY = {}

for dataset_name in datasets.keys():

    DATASET_REGISTRY[dataset_name] = {

        "data": datasets[dataset_name],

        "metadata": metadata[dataset_name],

        "feature_summary": feature_summary[dataset_name]

    }

print("Dataset registry created.")

Dataset registry created.


In [71]:
# ============================================================
# Cell 62 : Factory Helper Functions
# ============================================================

def get_model_factory(model_name):

    model_name = model_name.upper()

    if model_name not in BASELINE_MODELS:

        raise ValueError(f"{model_name} not registered.")

    return BASELINE_MODELS[model_name]["factory"]


def get_model_config(model_name):

    model_name = model_name.upper()

    if model_name not in BASELINE_MODELS:

        raise ValueError(f"{model_name} not registered.")

    return BASELINE_MODELS[model_name]["config"]


print("Factory helper functions created.")

Factory helper functions created.


In [72]:
# ============================================================
# Cell 63 : Dataset Helper Functions
# ============================================================

def get_dataset(dataset_name):

    return DATASET_REGISTRY[dataset_name]["data"]


def get_metadata(dataset_name):

    return DATASET_REGISTRY[dataset_name]["metadata"]


def get_feature_summary(dataset_name):

    return DATASET_REGISTRY[dataset_name]["feature_summary"]


print("Dataset helper functions created.")

Dataset helper functions created.


In [73]:
# ============================================================
# Cell 64 : Registry Summary
# ============================================================

summary = []

for model_name, info in BASELINE_MODELS.items():

    summary.append({

        "Model": model_name,

        "Datasets": len(info["models"]),

        "Factory": info["factory"].__name__,

        "Description": info["description"]

    })

registry_summary = pd.DataFrame(summary)

display(registry_summary)

,Model,Datasets,Factory,Description
0,CTGAN,3,create_ctgan,Conditional Tabular GAN
1,TVAE,3,create_tvae,Tabular Variational Autoencoder
2,DP_CTGAN,3,create_dp_ctgan,Differentially Private CTGAN


In [74]:
# ============================================================
# Cell 65 : Validation
# ============================================================

assert len(BASELINE_MODELS) == 3

assert len(DATASET_REGISTRY) == 3

for model_name in BASELINE_MODELS:

    assert callable(BASELINE_MODELS[model_name]["factory"])

for dataset_name in DATASET_REGISTRY:

    assert dataset_name in datasets

print("=" * 70)
print("Unified Registry Validation Successful")
print("=" * 70)

Unified Registry Validation Successful


In [75]:
# ============================================================
# Cell 66 : Export Registry
# ============================================================

registry_export = {

    "baseline_models": list(BASELINE_MODELS.keys()),

    "datasets": list(DATASET_REGISTRY.keys()),

    "device": DEVICE.type,

    "random_seed": RANDOM_STATE

}

with open(

    CONFIG_DIR / "baseline_registry.json",

    "w"

) as f:

    json.dump(

        registry_export,

        f,

        indent=4

    )

print("Registry exported successfully.")

Registry exported successfully.


In [76]:
# ============================================================
# Cell 67 : Final Verification
# ============================================================

print("=" * 90)
print("NOTEBOOK 04 : BASELINE MODEL DEFINITIONS")
print("=" * 90)

print()

print("Datasets")

for dataset in DATASET_REGISTRY:
    print(f"✓ {dataset}")

print()

print("Baseline Models")

for model in BASELINE_MODELS:
    print(f"✓ {model}")

print()

print("Metadata Generated :", len(metadata))

print("Configurations Saved :", len(MODEL_CONFIGS))

print("Training Status : NOT STARTED")

print("Registry Exported : YES")

print()

print("Notebook 04 Completed Successfully")

print()

print("Next Notebook")

print("Notebook 05 : Proposed SPP-GAN")

print("=" * 90)

NOTEBOOK 04 : BASELINE MODEL DEFINITIONS

Datasets
✓ Adult Income
✓ Bank Marketing
✓ Breast Cancer

Baseline Models
✓ CTGAN
✓ TVAE
✓ DP_CTGAN

Metadata Generated : 3
Configurations Saved : 4
Training Status : NOT STARTED
Registry Exported : YES

Notebook 04 Completed Successfully

Next Notebook
Notebook 05 : Proposed SPP-GAN


In [77]:
# ============================================================
# Cell 68 : Metadata Validation
# ============================================================

print("="*80)
print("VALIDATING METADATA")
print("="*80)

metadata_validation = []

for dataset_name, meta in metadata.items():

    try:

        columns = meta.to_dict()["columns"]

        metadata_validation.append({

            "Dataset": dataset_name,

            "Columns": len(columns),

            "Status": "Valid"

        })

        print(f"✓ {dataset_name} : {len(columns)} columns")

    except Exception as e:

        metadata_validation.append({

            "Dataset": dataset_name,

            "Columns": 0,

            "Status": "Invalid"

        })

        print(f"✗ {dataset_name} : {e}")

metadata_validation_df = pd.DataFrame(metadata_validation)

display(metadata_validation_df)

VALIDATING METADATA
✓ Adult Income : 111 columns
✓ Bank Marketing : 44 columns
✓ Breast Cancer : 31 columns


,Dataset,Columns,Status
0,Adult Income,111,Valid
1,Bank Marketing,44,Valid
2,Breast Cancer,31,Valid


In [78]:
METADATA_EXPORT_DIR = (
    BASELINE_MODEL_DIR /
    "metadata"
)

METADATA_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for dataset_name, meta in metadata.items():

    filename = (
        dataset_name.lower()
        .replace(" ", "_")
        + "_metadata.json"
    )

    meta.save_to_json(
        METADATA_EXPORT_DIR / filename,
        mode='overwrite'
    )

print("="*70)
print("Metadata exported successfully.")
print("="*70)

Metadata exported successfully.


In [79]:
# ============================================================
# Cell 70 : Metadata Summary
# ============================================================

metadata_summary = []

for dataset_name, meta in metadata.items():

    info = meta.to_dict()

    metadata_summary.append({

        "Dataset": dataset_name,

        "Number of Columns": len(info["columns"]),

        "Primary Key": info.get("primary_key", None)

    })

metadata_summary_df = pd.DataFrame(metadata_summary)

display(metadata_summary_df)

metadata_summary_df.to_csv(

    RESULT_DIR /
    "baseline_metadata_summary.csv",

    index=False

)

print("Metadata summary exported.")

,Dataset,Number of Columns,Primary Key
0,Adult Income,111,None
1,Bank Marketing,44,None
2,Breast Cancer,31,None


Metadata summary exported.


In [80]:
# ============================================================
# Cell 71 : Metadata Loader
# ============================================================

from sdv.metadata import SingleTableMetadata


def load_metadata(dataset_name):

    """
    Load metadata from JSON.
    """

    filename = (

        dataset_name.lower()

        .replace(" ", "_")

        + "_metadata.json"

    )

    metadata_path = (

        METADATA_EXPORT_DIR /

        filename

    )

    meta = SingleTableMetadata()

    meta.load_from_json(metadata_path)

    return meta


print("Metadata loader created.")

Metadata loader created.


In [81]:
# ============================================================
# Cell 72 : Verify Exported Metadata
# ============================================================

print("="*80)
print("VERIFYING EXPORTED METADATA")
print("="*80)

for dataset_name in metadata.keys():

    meta = load_metadata(dataset_name)

    print(

        f"✓ {dataset_name:<20}"

        f"{len(meta.columns)} columns"

    )

VERIFYING EXPORTED METADATA
✓ Adult Income        0 columns
✓ Bank Marketing      0 columns
✓ Breast Cancer       0 columns


In [82]:
# ============================================================
# Cell 73 : Metadata Registry
# ============================================================

METADATA_REGISTRY = {

    dataset_name: {

        "metadata": metadata[dataset_name],

        "json_file":

            METADATA_EXPORT_DIR /

            f"{dataset_name.lower().replace(' ','_')}_metadata.json"

    }

    for dataset_name in metadata

}

print("Metadata registry created.")

Metadata registry created.


In [83]:
# ============================================================
# Cell 74 : Section Verification
# ============================================================

print("="*90)

print("NOTEBOOK 04")

print("SECTION I COMPLETED")

print("="*90)

print()

print("Metadata Objects")

for dataset in METADATA_REGISTRY:

    print(f"✓ {dataset}")

print()

print("JSON Files")

for dataset, info in METADATA_REGISTRY.items():

    print(f"✓ {info['json_file'].name}")

print()

print("Metadata Export : SUCCESS")

print("Metadata Validation : SUCCESS")

print()

print("Next")

print("Notebook 05 : Proposed SPP-GAN")

print("="*90)

NOTEBOOK 04
SECTION I COMPLETED

Metadata Objects
✓ Adult Income
✓ Bank Marketing
✓ Breast Cancer

JSON Files
✓ adult_income_metadata.json
✓ bank_marketing_metadata.json
✓ breast_cancer_metadata.json

Metadata Export : SUCCESS
Metadata Validation : SUCCESS

Next
Notebook 05 : Proposed SPP-GAN


In [84]:
# ============================================================
# Cell 75 : Validate Datasets
# ============================================================

print("=" * 80)
print("VALIDATING DATASETS")
print("=" * 80)

for name, df in datasets.items():

    assert isinstance(df, pd.DataFrame)

    assert len(df) > 0

    assert len(df.columns) > 0

    print(f"✓ {name:<20} {df.shape}")

print("\nDataset validation successful.")

VALIDATING DATASETS
✓ Adult Income         (48813, 111)
✓ Bank Marketing       (45211, 44)
✓ Breast Cancer        (569, 31)

Dataset validation successful.


In [85]:
# ============================================================
# Cell 76 : Validate Metadata
# ============================================================

print("=" * 80)
print("VALIDATING METADATA")
print("=" * 80)

for name, meta in metadata.items():

    assert meta is not None

    assert len(meta.to_dict()["columns"]) > 0

    print(

        f"✓ {name:<20}"

        f"{len(meta.to_dict()['columns'])} columns"

    )

print("\nMetadata validation successful.")

VALIDATING METADATA
✓ Adult Income        111 columns
✓ Bank Marketing      44 columns
✓ Breast Cancer       31 columns

Metadata validation successful.


In [86]:
# ============================================================
# Cell 77 : Validate Configurations
# ============================================================

print("=" * 80)
print("VALIDATING CONFIGURATIONS")
print("=" * 80)

configs = {

    "CTGAN": CTGAN_CONFIG,

    "TVAE": TVAE_CONFIG,

    "DP-CTGAN": DP_CTGAN_FACTORY_CONFIG

}

for name, cfg in configs.items():

    assert isinstance(cfg, dict)

    assert len(cfg) > 0

    print(f"✓ {name}")

print("\nConfiguration validation successful.")

VALIDATING CONFIGURATIONS
✓ CTGAN
✓ TVAE
✓ DP-CTGAN

Configuration validation successful.


In [87]:
# ============================================================
# Cell 78 : Validate Factory Functions
# ============================================================

print("=" * 80)
print("VALIDATING FACTORIES")
print("=" * 80)

factories = {

    "CTGAN": create_ctgan,

    "TVAE": create_tvae,

    "DP-CTGAN": create_dp_ctgan

}

for name, factory in factories.items():

    assert callable(factory)

    print(f"✓ {name}")

print("\nFactory validation successful.")

VALIDATING FACTORIES
✓ CTGAN
✓ TVAE
✓ DP-CTGAN

Factory validation successful.


In [88]:
# ============================================================
# Cell 79 : Validate Registries
# ============================================================

print("=" * 80)
print("VALIDATING REGISTRIES")
print("=" * 80)

assert len(BASELINE_MODELS) == 3

assert len(DATASET_REGISTRY) == 3

assert len(MODEL_CONFIGS) == 4

print("✓ Baseline Registry")

print("✓ Dataset Registry")

print("✓ Model Registry")

print("\nRegistry validation successful.")

VALIDATING REGISTRIES
✓ Baseline Registry
✓ Dataset Registry
✓ Model Registry

Registry validation successful.


In [89]:
# ============================================================
# Cell 80 : Validate Exported Files
# ============================================================

print("=" * 80)
print("VALIDATING EXPORTED FILES")
print("=" * 80)

required_files = [

    CONFIG_DIR / "project_config.json",

    CONFIG_DIR / "baseline_registry.json",

    CONFIG_DIR / "baseline_models" / "ctgan.json",

    CONFIG_DIR / "baseline_models" / "tvae.json",

    CONFIG_DIR / "baseline_models" / "dp_ctgan.json"

]

for file in required_files:

    assert file.exists(), f"Missing: {file}"

    print(f"✓ {file.name}")

print("\nConfiguration files validated.")

VALIDATING EXPORTED FILES
✓ project_config.json
✓ baseline_registry.json
✓ ctgan.json
✓ tvae.json
✓ dp_ctgan.json

Configuration files validated.


In [90]:
# ============================================================
# Cell 81 : Validate Metadata Files
# ============================================================

print("=" * 80)
print("VALIDATING METADATA FILES")
print("=" * 80)

for dataset in metadata.keys():

    file = (

        METADATA_EXPORT_DIR /

        f"{dataset.lower().replace(' ','_')}_metadata.json"

    )

    assert file.exists()

    print(f"✓ {file.name}")

print("\nMetadata files validated.")

VALIDATING METADATA FILES
✓ adult_income_metadata.json
✓ bank_marketing_metadata.json
✓ breast_cancer_metadata.json

Metadata files validated.


In [91]:
# ============================================================
# Cell 82 : Notebook Readiness
# ============================================================

print("=" * 80)
print("NOTEBOOK READINESS")
print("=" * 80)

checks = {

    "Datasets": len(datasets),

    "Metadata": len(metadata),

    "Baseline Models": len(BASELINE_MODELS),

    "Configurations": len(MODEL_CONFIGS),

    "Factories": len(factories)

}

for key, value in checks.items():

    print(f"{key:<20}: {value}")

print("\nNotebook ready.")

NOTEBOOK READINESS
Datasets            : 3
Metadata            : 3
Baseline Models     : 3
Configurations      : 4
Factories           : 3

Notebook ready.


In [92]:
# ============================================================
# Cell 83 : Notebook Completion
# ============================================================

print("=" * 100)

print("NOTEBOOK 04 : BASELINE MODEL DEFINITIONS")

print("=" * 100)

print()

print("STATUS")

print("✓ Environment Ready")

print("✓ Datasets Loaded")

print("✓ Metadata Generated")

print("✓ Hyperparameters Registered")

print("✓ CTGAN Factory")

print("✓ TVAE Factory")

print("✓ DP-CTGAN Factory")

print("✓ Unified Registry")

print("✓ Metadata Exported")

print("✓ Validation Passed")

print()

print("OUTPUTS")

print(f"Datasets           : {len(datasets)}")

print(f"Metadata           : {len(metadata)}")

print(f"Baseline Models    : {len(BASELINE_MODELS)}")

print(f"Configurations     : {len(MODEL_CONFIGS)}")

print()

print("Notebook 04 COMPLETED SUCCESSFULLY")

print()

print("NEXT NOTEBOOK")

print("Notebook 05 : Proposed SPP-GAN")

print()

print("Then")

print("Notebook 06 : Unified Model Training")

print("=" * 100)

NOTEBOOK 04 : BASELINE MODEL DEFINITIONS

STATUS
✓ Environment Ready
✓ Datasets Loaded
✓ Metadata Generated
✓ Hyperparameters Registered
✓ CTGAN Factory
✓ TVAE Factory
✓ DP-CTGAN Factory
✓ Unified Registry
✓ Metadata Exported
✓ Validation Passed

OUTPUTS
Datasets           : 3
Metadata           : 3
Baseline Models    : 3
Configurations     : 4

Notebook 04 COMPLETED SUCCESSFULLY

NEXT NOTEBOOK
Notebook 05 : Proposed SPP-GAN

Then
Notebook 06 : Unified Model Training


In [93]:
# ============================================================
# Cell 84 : Verify Project Structure
# ============================================================

print("=" * 90)
print("VERIFYING PROJECT STRUCTURE")
print("=" * 90)

required_dirs = {
    "Project Root": PROJECT_ROOT,
    "Config": CONFIG_DIR,
    "Datasets": DATASET_DIR,
    "Processed Data": PROCESSED_DIR,
    "Baseline Models": BASELINE_MODEL_DIR,
    "Metadata": METADATA_EXPORT_DIR,
    "Results": RESULT_DIR
}

verification = []

for name, path in required_dirs.items():

    exists = path.exists()

    verification.append({
        "Directory": name,
        "Exists": exists
    })

    print(f"{'✓' if exists else '✗'} {name:<20} {path}")

project_structure_df = pd.DataFrame(verification)

display(project_structure_df)

VERIFYING PROJECT STRUCTURE
✓ Project Root         /content/drive/MyDrive/SPP_GAN_Project
✓ Config               /content/drive/MyDrive/SPP_GAN_Project/configs
✓ Datasets             /content/drive/MyDrive/SPP_GAN_Project/datasets
✓ Processed Data       /content/drive/MyDrive/SPP_GAN_Project/datasets/processed
✓ Baseline Models      /content/drive/MyDrive/SPP_GAN_Project/models/baseline_models
✓ Metadata             /content/drive/MyDrive/SPP_GAN_Project/models/baseline_models/metadata
✓ Results              /content/drive/MyDrive/SPP_GAN_Project/results


,Directory,Exists
0,Project Root,True
1,Config,True
2,Datasets,True
3,Processed Data,True
4,Baseline Models,True
5,Metadata,True
6,Results,True


In [94]:
# ============================================================
# Cell 85 : Verify Model Factories
# ============================================================

print("=" * 90)
print("VERIFYING FACTORY FUNCTIONS")
print("=" * 90)

factories = {
    "CTGAN": create_ctgan,
    "TVAE": create_tvae,
    "DP-CTGAN": create_dp_ctgan
}

for name, func in factories.items():

    assert callable(func)

    print(f"✓ {name}")

print("\nAll factories verified.")

VERIFYING FACTORY FUNCTIONS
✓ CTGAN
✓ TVAE
✓ DP-CTGAN

All factories verified.


In [95]:
# ============================================================
# Cell 86 : Verify Registries
# ============================================================

print("=" * 90)
print("VERIFYING REGISTRIES")
print("=" * 90)

assert len(BASELINE_MODELS) == 3
assert len(DATASET_REGISTRY) == 3
assert len(METADATA_REGISTRY) == 3

print("✓ BASELINE_MODELS")
print("✓ DATASET_REGISTRY")
print("✓ METADATA_REGISTRY")

print("\nRegistry verification successful.")

VERIFYING REGISTRIES
✓ BASELINE_MODELS
✓ DATASET_REGISTRY
✓ METADATA_REGISTRY

Registry verification successful.


In [96]:
# ============================================================
# Cell 87 : Verify Exported Files
# ============================================================

print("=" * 90)
print("VERIFYING EXPORTED FILES")
print("=" * 90)

files = [

    CONFIG_DIR / "project_config.json",

    CONFIG_DIR / "baseline_registry.json",

    CONFIG_DIR / "baseline_models" / "ctgan.json",

    CONFIG_DIR / "baseline_models" / "tvae.json",

    CONFIG_DIR / "baseline_models" / "dp_ctgan.json"

]

for file in files:

    if file.exists():

        print(f"✓ {file.name}")

    else:

        print(f"✗ {file.name}")

VERIFYING EXPORTED FILES
✓ project_config.json
✓ baseline_registry.json
✓ ctgan.json
✓ tvae.json
✓ dp_ctgan.json


In [97]:
# ============================================================
# Cell 88 : Verify Metadata Files
# ============================================================

print("=" * 90)
print("VERIFYING METADATA FILES")
print("=" * 90)

for dataset in metadata.keys():

    filename = (

        dataset.lower()

        .replace(" ", "_")

        + "_metadata.json"

    )

    file = METADATA_EXPORT_DIR / filename

    if file.exists():

        print(f"✓ {filename}")

    else:

        print(f"✗ {filename}")

VERIFYING METADATA FILES
✓ adult_income_metadata.json
✓ bank_marketing_metadata.json
✓ breast_cancer_metadata.json


In [98]:
# ============================================================
# Cell 89 : Notebook Summary
# ============================================================

summary = {

    "Datasets": len(datasets),

    "Metadata": len(metadata),

    "Baseline Models": len(BASELINE_MODELS),

    "Configurations": len(MODEL_CONFIGS),

    "Factories": len(factories),

    "Registries": 3

}

summary_df = pd.DataFrame(

    summary.items(),

    columns=["Component", "Count"]

)

display(summary_df)

,Component,Count
0,Datasets,3
1,Metadata,3
2,Baseline Models,3
3,Configurations,4
4,Factories,3
5,Registries,3


In [99]:
# ============================================================
# Cell 90 : Final Notebook Verification
# ============================================================

print("=" * 100)

print("NOTEBOOK 04 : BASELINE MODEL DEFINITIONS")

print("=" * 100)

print()

print("STATUS")

print("-----------------------------------------------")

print("✓ Environment Configured")

print("✓ Processed Datasets Loaded")

print("✓ Metadata Generated")

print("✓ Hyperparameter Registry Created")

print("✓ CTGAN Factory")

print("✓ TVAE Factory")

print("✓ DP-CTGAN Factory")

print("✓ Unified Model Registry")

print("✓ Metadata Exported")

print("✓ Configuration Files Exported")

print("✓ Validation Successful")

print()

print("OUTPUT SUMMARY")

print("-----------------------------------------------")

print(f"Datasets             : {len(datasets)}")

print(f"Metadata             : {len(metadata)}")

print(f"Baseline Models      : {len(BASELINE_MODELS)}")

print(f"Configurations       : {len(MODEL_CONFIGS)}")

print()

print("READY FOR")

print("-----------------------------------------------")

print("✓ Notebook 05 : Proposed SPP-GAN")

print("✓ Notebook 06 : Unified Model Training")

print()

print("Notebook 04 COMPLETED SUCCESSFULLY")

print("=" * 100)

NOTEBOOK 04 : BASELINE MODEL DEFINITIONS

STATUS
-----------------------------------------------
✓ Environment Configured
✓ Processed Datasets Loaded
✓ Metadata Generated
✓ Hyperparameter Registry Created
✓ CTGAN Factory
✓ TVAE Factory
✓ DP-CTGAN Factory
✓ Unified Model Registry
✓ Metadata Exported
✓ Configuration Files Exported
✓ Validation Successful

OUTPUT SUMMARY
-----------------------------------------------
Datasets             : 3
Metadata             : 3
Baseline Models      : 3
Configurations       : 4

READY FOR
-----------------------------------------------
✓ Notebook 05 : Proposed SPP-GAN
✓ Notebook 06 : Unified Model Training

Notebook 04 COMPLETED SUCCESSFULLY


In [100]:
# ============================================================
# FINAL METADATA VERIFICATION
# ============================================================

from sdv.metadata import SingleTableMetadata

print("=" * 80)
print("VERIFYING SAVED METADATA")
print("=" * 80)

for dataset_name, dataframe in datasets.items():

    filename = (
        dataset_name.lower().replace(" ", "_")
        + "_metadata.json"
    )

    metadata_path = METADATA_DIR / filename

    loaded_metadata = SingleTableMetadata.load_from_json(
        filepath=str(metadata_path)
    )

    dataset_columns = set(dataframe.columns)

    metadata_columns = set(loaded_metadata.columns.keys())

    print(f"\n{dataset_name}")

    print(f"Dataset Columns  : {len(dataset_columns)}")
    print(f"Metadata Columns : {len(metadata_columns)}")

    if dataset_columns == metadata_columns:

        print("✓ PERFECT MATCH")

    else:

        print("✗ MISMATCH")

        print("Missing:",
              dataset_columns - metadata_columns)

        print("Extra:",
              metadata_columns - dataset_columns)

VERIFYING SAVED METADATA

Adult Income
Dataset Columns  : 111
Metadata Columns : 111
✓ PERFECT MATCH

Bank Marketing
Dataset Columns  : 44
Metadata Columns : 44
✓ PERFECT MATCH

Breast Cancer
Dataset Columns  : 31
Metadata Columns : 31
✓ PERFECT MATCH
